# 04 — Modelagem analítica Gold

## Objetivo

Este notebook constrói o modelo analítico final para consumo por ferramentas de BI.

O modelo utiliza uma estrutura dimensional composta por:

### Dimensões

- `dim_data`;
- `dim_clientes`;
- `dim_produtos`;
- `dim_vendedores`;
- `dim_canais`;
- `dim_regioes`.

### Fatos

- `fct_pedidos`: um registro por pedido;
- `fct_pedidos_itens`: um registro por item do pedido;
- `fct_entregas`: um registro por entrega;
- `fct_ocorrencias`: um registro por ocorrência.

### Agregação analítica

- `agg_indicadores_mensais`: indicadores mensais preparados para consumo pelo BI.

As tabelas Gold utilizam chaves técnicas determinísticas e um membro
`NAO_IDENTIFICADO` para preservar registros que possuam relacionamentos ausentes
ou inválidos na origem.

In [0]:
# ============================================================
# CONFIGURAÇÕES
# ============================================================

from datetime import datetime, timezone
from functools import reduce
import uuid

from pyspark.sql import DataFrame
from pyspark.sql import functions as F

spark.conf.set(
    "spark.sql.session.timeZone",
    "America/Sao_Paulo"
)

SCHEMA_SILVER = "case_data_engineer_silver"
SCHEMA_GOLD = "case_data_engineer_gold"

GOLD_RUN_ID = str(uuid.uuid4())
GOLD_TIMESTAMP = datetime.now(timezone.utc)

print(f"Gold run ID: {GOLD_RUN_ID}")
print(f"Timestamp UTC: {GOLD_TIMESTAMP}")

In [0]:
# ============================================================
# LOCALIZAÇÃO DO CATÁLOGO
# ============================================================

catalogos_disponiveis = [
    linha["catalog"]
    for linha in spark.sql("SHOW CATALOGS").collect()
]

catalogos_ignorados = {
    "system",
    "samples",
    "hive_metastore"
}

CATALOG = None

for catalogo in catalogos_disponiveis:

    if catalogo.lower() in catalogos_ignorados:
        continue

    try:
        (
            spark.table(
                f"`{catalogo}`."
                f"`{SCHEMA_SILVER}`."
                f"`silver_pedidos`"
            )
            .limit(1)
            .collect()
        )

        CATALOG = catalogo
        break

    except Exception:
        continue

if CATALOG is None:
    raise RuntimeError(
        "A camada Silver não foi localizada, execute os notebooks 02 e 03 antes da Gold"
    )

spark.sql(
    f"""
    CREATE SCHEMA IF NOT EXISTS
    `{CATALOG}`.`{SCHEMA_GOLD}`
    COMMENT 'Modelo analítico Gold do case de Engenharia de Dados'
    """
)

print(f"Catálogo: {CATALOG}")
print(f"Silver: {CATALOG}.{SCHEMA_SILVER}")
print(f"Gold: {CATALOG}.{SCHEMA_GOLD}")

In [0]:
# ============================================================
# LEITURA DAS TABELAS SILVER
# ============================================================

silver_regioes = spark.table(
    f"`{CATALOG}`.`{SCHEMA_SILVER}`.`silver_regioes`"
)

silver_canais = spark.table(
    f"`{CATALOG}`.`{SCHEMA_SILVER}`.`silver_canais`"
)

silver_clientes = spark.table(
    f"`{CATALOG}`.`{SCHEMA_SILVER}`.`silver_clientes`"
)

silver_vendedores = spark.table(
    f"`{CATALOG}`.`{SCHEMA_SILVER}`.`silver_vendedores`"
)

silver_produtos = spark.table(
    f"`{CATALOG}`.`{SCHEMA_SILVER}`.`silver_produtos`"
)

silver_pedidos = spark.table(
    f"`{CATALOG}`.`{SCHEMA_SILVER}`.`silver_pedidos`"
)

silver_pedidos_itens = spark.table(
    f"`{CATALOG}`.`{SCHEMA_SILVER}`.`silver_pedidos_itens`"
)

silver_entregas = spark.table(
    f"`{CATALOG}`.`{SCHEMA_SILVER}`.`silver_entregas`"
)

silver_ocorrencias = spark.table(
    f"`{CATALOG}`.`{SCHEMA_SILVER}`.`silver_ocorrencias`"
)

tabelas_silver = {
    "regioes": silver_regioes,
    "canais": silver_canais,
    "clientes": silver_clientes,
    "vendedores": silver_vendedores,
    "produtos": silver_produtos,
    "pedidos": silver_pedidos,
    "pedidos_itens": silver_pedidos_itens,
    "entregas": silver_entregas,
    "ocorrencias": silver_ocorrencias
}

for nome, dataframe in tabelas_silver.items():
    print(
        f"{nome:<18}: "
        f"{dataframe.count()} registros"
    )

In [0]:
# ============================================================
# FUNÇÕES AUXILIARES
# ============================================================

# Geramos uma chave técnica determinística utilizando SHA-256 e a chave 0 é reservada para o membro não identificado
def gerar_chave_tecnica(coluna):
    valor = F.upper(
        F.trim(
            coluna.cast("string")
        )
    )

    return (
        F.when(
            valor.isNull()
            | (valor == ""),
            F.lit("0")
        )
        .otherwise(
            F.sha2(valor, 256)
        )
    )

# Retorna uma expressão booleana indicando se alguma coluna dq_* do registro está marcada como verdadeira
def expressao_tem_dq(dataframe):
    colunas_dq = [
        coluna
        for coluna in dataframe.columns
        if coluna.startswith("dq_")
    ]

    if not colunas_dq:
        return F.lit(False)

    return reduce(
        lambda anterior, atual: anterior | atual,
        [
            F.coalesce(
                F.col(coluna),
                F.lit(False)
            )
            for coluna in colunas_dq
        ]
    )


def salvar_tabela_gold(
    dataframe,
    nome_tabela
):
    tabela_completa = (
        f"`{CATALOG}`."
        f"`{SCHEMA_GOLD}`."
        f"`{nome_tabela}`"
    )

    (
        dataframe.write
        .format("delta")
        .mode("overwrite")
        .option(
            "overwriteSchema",
            "true"
        )
        .saveAsTable(
            tabela_completa
        )
    )

    quantidade_persistida = (
        spark.table(tabela_completa)
        .count()
    )

    print(
        f"{nome_tabela}: "
        f"{quantidade_persistida} registros persistidos"
    )


print("Funções auxiliares Gold disponíveis")

In [0]:
# ============================================================
# DIMENSÃO GOLD — REGIÕES
# ============================================================

dim_regioes_negocio = (
    silver_regioes
    .select(
        gerar_chave_tecnica(
            F.col("regional_code")
        ).alias(
            "region_key"
        ),
        F.col("regional_code"),
        F.col("regional_name"),
        F.col("state_code"),
        F.col("manager_name"),
        F.col("is_active"),
        expressao_tem_dq(
            silver_regioes
        ).alias(
            "has_data_quality_issue"
        )
    )
)

dim_regioes_desconhecida = (
    spark.range(1)
    .select(
        F.lit("0").alias(
            "region_key"
        ),
        F.lit("NAO_IDENTIFICADO").alias(
            "regional_code"
        ),
        F.lit("Não identificada").alias(
            "regional_name"
        ),
        F.lit(None).cast("string").alias(
            "state_code"
        ),
        F.lit(None).cast("string").alias(
            "manager_name"
        ),
        F.lit(None).cast("boolean").alias(
            "is_active"
        ),
        F.lit(True).alias(
            "has_data_quality_issue"
        )
    )
)

dim_regioes = (
    dim_regioes_desconhecida
    .unionByName(
        dim_regioes_negocio
    )
    .withColumn(
        "_gold_run_id",
        F.lit(GOLD_RUN_ID)
    )
    .withColumn(
        "_gold_processed_at_utc",
        F.lit(GOLD_TIMESTAMP)
    )
)

salvar_tabela_gold(
    dim_regioes,
    "dim_regioes"
)

display(
    dim_regioes.orderBy("regional_code")
)

In [0]:
# ============================================================
# DIMENSÃO GOLD — CANAIS
# ============================================================

dim_canais_negocio = (
    silver_canais
    .select(
        gerar_chave_tecnica(
            F.col("channel_id")
        ).alias(
            "channel_key"
        ),
        F.col("channel_id"),
        F.col("channel_name"),
        F.col("channel_type"),
        F.col("is_active"),
        expressao_tem_dq(
            silver_canais
        ).alias(
            "has_data_quality_issue"
        )
    )
)

dim_canais_desconhecida = (
    spark.range(1)
    .select(
        F.lit("0").alias(
            "channel_key"
        ),
        F.lit("NAO_IDENTIFICADO").alias(
            "channel_id"
        ),
        F.lit("Não identificado").alias(
            "channel_name"
        ),
        F.lit("NAO_IDENTIFICADO").alias(
            "channel_type"
        ),
        F.lit(None).cast("boolean").alias(
            "is_active"
        ),
        F.lit(True).alias(
            "has_data_quality_issue"
        )
    )
)

dim_canais = (
    dim_canais_desconhecida
    .unionByName(
        dim_canais_negocio
    )
    .withColumn(
        "_gold_run_id",
        F.lit(GOLD_RUN_ID)
    )
    .withColumn(
        "_gold_processed_at_utc",
        F.lit(GOLD_TIMESTAMP)
    )
)

salvar_tabela_gold(
    dim_canais,
    "dim_canais"
)

display(
    dim_canais.orderBy("channel_id")
)

In [0]:
# ============================================================
# DIMENSÃO GOLD — CLIENTES
# ============================================================

dim_clientes_negocio = (
    silver_clientes
    .select(
        gerar_chave_tecnica(
            F.col("customer_id")
        ).alias(
            "customer_key"
        ),
        F.col("customer_id"),
        F.col("customer_name"),
        F.col("segment"),
        F.col("customer_size"),
        F.col("city_name"),
        F.col("state_code"),
        F.col("registration_date"),
        F.col("email"),
        F.col("customer_status"),
        expressao_tem_dq(
            silver_clientes
        ).alias(
            "has_data_quality_issue"
        )
    )
)

dim_clientes_desconhecida = (
    spark.range(1)
    .select(
        F.lit("0").alias(
            "customer_key"
        ),
        F.lit("NAO_IDENTIFICADO").alias(
            "customer_id"
        ),
        F.lit("Cliente não identificado").alias(
            "customer_name"
        ),
        F.lit("NAO_IDENTIFICADO").alias(
            "segment"
        ),
        F.lit("NAO_IDENTIFICADO").alias(
            "customer_size"
        ),
        F.lit(None).cast("string").alias(
            "city_name"
        ),
        F.lit(None).cast("string").alias(
            "state_code"
        ),
        F.lit(None).cast("date").alias(
            "registration_date"
        ),
        F.lit(None).cast("string").alias(
            "email"
        ),
        F.lit("NAO_IDENTIFICADO").alias(
            "customer_status"
        ),
        F.lit(True).alias(
            "has_data_quality_issue"
        )
    )
)

dim_clientes = (
    dim_clientes_desconhecida
    .unionByName(
        dim_clientes_negocio
    )
    .withColumn(
        "_gold_run_id",
        F.lit(GOLD_RUN_ID)
    )
    .withColumn(
        "_gold_processed_at_utc",
        F.lit(GOLD_TIMESTAMP)
    )
)

salvar_tabela_gold(
    dim_clientes,
    "dim_clientes"
)

display(
    dim_clientes.orderBy("customer_id")
)

In [0]:
# ============================================================
# DIMENSÃO GOLD — PRODUTOS
# ============================================================

dim_produtos_negocio = (
    silver_produtos
    .select(
        gerar_chave_tecnica(
            F.col("product_id")
        ).alias(
            "product_key"
        ),
        F.col("product_id"),
        F.col("product_name"),
        F.col("category"),
        F.col("subcategory"),
        F.col("product_status"),
        F.col("list_price"),
        F.col("currency_code"),
        F.col("product_family"),
        F.col("tags"),
        expressao_tem_dq(
            silver_produtos
        ).alias(
            "has_data_quality_issue"
        )
    )
)

dim_produtos_desconhecida = (
    spark.range(1)
    .select(
        F.lit("0").alias(
            "product_key"
        ),
        F.lit("NAO_IDENTIFICADO").alias(
            "product_id"
        ),
        F.lit("Produto não identificado").alias(
            "product_name"
        ),
        F.lit("NAO_IDENTIFICADO").alias(
            "category"
        ),
        F.lit("NAO_IDENTIFICADO").alias(
            "subcategory"
        ),
        F.lit("NAO_IDENTIFICADO").alias(
            "product_status"
        ),
        F.lit(None).cast(
            "decimal(18,2)"
        ).alias(
            "list_price"
        ),
        F.lit("BRL").alias(
            "currency_code"
        ),
        F.lit("NAO_IDENTIFICADO").alias(
            "product_family"
        ),
        F.expr(
            "CAST(array() AS ARRAY<STRING>)"
        ).alias(
            "tags"
        ),
        F.lit(True).alias(
            "has_data_quality_issue"
        )
    )
)

dim_produtos = (
    dim_produtos_desconhecida
    .unionByName(
        dim_produtos_negocio
    )
    .withColumn(
        "_gold_run_id",
        F.lit(GOLD_RUN_ID)
    )
    .withColumn(
        "_gold_processed_at_utc",
        F.lit(GOLD_TIMESTAMP)
    )
)

salvar_tabela_gold(
    dim_produtos,
    "dim_produtos"
)

display(
    dim_produtos.orderBy("product_id")
)

In [0]:
# ============================================================
# DIMENSÃO GOLD — VENDEDORES
# ============================================================

dim_vendedores_negocio = (
    silver_vendedores
    .select(
        gerar_chave_tecnica(
            F.col("seller_id")
        ).alias(
            "seller_key"
        ),
        F.col("seller_id"),
        F.col("seller_name"),

        F.when(
            F.col("dq_missing_channel")
            | F.col("dq_orphan_channel"),
            F.lit("0")
        )
        .otherwise(
            gerar_chave_tecnica(
                F.col("channel_id")
            )
        )
        .alias(
            "channel_key"
        ),

        F.col("channel_id"),

        F.when(
            F.col("dq_missing_region")
            | F.col("dq_orphan_region"),
            F.lit("0")
        )
        .otherwise(
            gerar_chave_tecnica(
                F.col("regional_code")
            )
        )
        .alias(
            "region_key"
        ),

        F.col("regional_code"),
        F.col("hire_date"),
        F.col("seller_status"),

        expressao_tem_dq(
            silver_vendedores
        ).alias(
            "has_data_quality_issue"
        )
    )
)

dim_vendedores_desconhecida = (
    spark.range(1)
    .select(
        F.lit("0").alias(
            "seller_key"
        ),
        F.lit("NAO_IDENTIFICADO").alias(
            "seller_id"
        ),
        F.lit("Vendedor não identificado").alias(
            "seller_name"
        ),
        F.lit("0").alias(
            "channel_key"
        ),
        F.lit("NAO_IDENTIFICADO").alias(
            "channel_id"
        ),
        F.lit("0").alias(
            "region_key"
        ),
        F.lit("NAO_IDENTIFICADO").alias(
            "regional_code"
        ),
        F.lit(None).cast("date").alias(
            "hire_date"
        ),
        F.lit("NAO_IDENTIFICADO").alias(
            "seller_status"
        ),
        F.lit(True).alias(
            "has_data_quality_issue"
        )
    )
)

dim_vendedores = (
    dim_vendedores_desconhecida
    .unionByName(
        dim_vendedores_negocio
    )
    .withColumn(
        "_gold_run_id",
        F.lit(GOLD_RUN_ID)
    )
    .withColumn(
        "_gold_processed_at_utc",
        F.lit(GOLD_TIMESTAMP)
    )
)

salvar_tabela_gold(
    dim_vendedores,
    "dim_vendedores"
)

display(
    dim_vendedores.orderBy("seller_id")
)

In [0]:
# ============================================================
# DIMENSÃO GOLD — DATA
# ============================================================

datas_modelo = (
    silver_pedidos
    .select(
        F.col("order_date").alias("calendar_date")
    )
    .unionByName(
        silver_pedidos.select(
            F.col("promised_date").alias("calendar_date")
        )
    )
    .unionByName(
        silver_entregas.select(
            F.to_date("shipped_at").alias("calendar_date")
        )
    )
    .unionByName(
        silver_entregas.select(
            F.to_date("delivered_at").alias("calendar_date")
        )
    )
    .unionByName(
        silver_ocorrencias.select(
            F.to_date("opened_at").alias("calendar_date")
        )
    )
    .unionByName(
        silver_ocorrencias.select(
            F.to_date("closed_at").alias("calendar_date")
        )
    )
    .filter(
        F.col("calendar_date").isNotNull()
    )
)

limites_calendario = (
    datas_modelo
    .agg(
        F.min("calendar_date").alias("min_date"),
        F.max("calendar_date").alias("max_date")
    )
    .first()
)

data_minima = limites_calendario["min_date"]
data_maxima = limites_calendario["max_date"]

if data_minima is None or data_maxima is None:
    raise RuntimeError(
        "Não foi possível determinar o intervalo da dimensão data"
    )

print(f"Data mínima: {data_minima}")
print(f"Data máxima: {data_maxima}")

In [0]:
nomes_meses = F.create_map(
    F.lit(1), F.lit("Janeiro"),
    F.lit(2), F.lit("Fevereiro"),
    F.lit(3), F.lit("Março"),
    F.lit(4), F.lit("Abril"),
    F.lit(5), F.lit("Maio"),
    F.lit(6), F.lit("Junho"),
    F.lit(7), F.lit("Julho"),
    F.lit(8), F.lit("Agosto"),
    F.lit(9), F.lit("Setembro"),
    F.lit(10), F.lit("Outubro"),
    F.lit(11), F.lit("Novembro"),
    F.lit(12), F.lit("Dezembro")
)

nomes_dias_semana = F.create_map(
    F.lit(1), F.lit("Domingo"),
    F.lit(2), F.lit("Segunda-feira"),
    F.lit(3), F.lit("Terça-feira"),
    F.lit(4), F.lit("Quarta-feira"),
    F.lit(5), F.lit("Quinta-feira"),
    F.lit(6), F.lit("Sexta-feira"),
    F.lit(7), F.lit("Sábado")
)

dim_data_negocio = (
    spark.range(1)
    .select(
        F.explode(
            F.sequence(
                F.lit(data_minima),
                F.lit(data_maxima),
                F.expr("INTERVAL 1 DAY")
            )
        ).alias(
            "calendar_date"
        )
    )
    .withColumn(
        "date_key",
        F.date_format(
            "calendar_date",
            "yyyyMMdd"
        ).cast("int")
    )
    .withColumn(
        "year_number",
        F.year("calendar_date")
    )
    .withColumn(
        "semester_number",
        F.when(
            F.month("calendar_date") <= 6,
            F.lit(1)
        ).otherwise(
            F.lit(2)
        )
    )
    .withColumn(
        "quarter_number",
        F.quarter("calendar_date")
    )
    .withColumn(
        "month_number",
        F.month("calendar_date")
    )
    .withColumn(
        "month_name",
        nomes_meses[
            F.col("month_number")
        ]
    )
    .withColumn(
        "year_month",
        F.date_format(
            "calendar_date",
            "yyyy-MM"
        )
    )
    .withColumn(
        "year_month_label",
        F.concat(
            F.col("month_name"),
            F.lit("/"),
            F.col("year_number")
        )
    )
    .withColumn(
        "day_number",
        F.dayofmonth("calendar_date")
    )
    .withColumn(
        "week_of_year",
        F.weekofyear("calendar_date")
    )
    .withColumn(
        "day_of_week_number",
        F.dayofweek("calendar_date")
    )
    .withColumn(
        "day_of_week_name",
        nomes_dias_semana[
            F.col("day_of_week_number")
        ]
    )
    .withColumn(
        "is_weekend",
        F.col("day_of_week_number").isin(
            1,
            7
        )
    )
)

dim_data_desconhecida = (
    spark.range(1)
    .select(
        F.lit(0).cast("int").alias(
            "date_key"
        ),
        F.lit(None).cast("date").alias(
            "calendar_date"
        ),
        F.lit(None).cast("int").alias(
            "year_number"
        ),
        F.lit(None).cast("int").alias(
            "semester_number"
        ),
        F.lit(None).cast("int").alias(
            "quarter_number"
        ),
        F.lit(None).cast("int").alias(
            "month_number"
        ),
        F.lit("Não identificado").alias(
            "month_name"
        ),
        F.lit("NAO_IDENTIFICADO").alias(
            "year_month"
        ),
        F.lit("Não identificado").alias(
            "year_month_label"
        ),
        F.lit(None).cast("int").alias(
            "day_number"
        ),
        F.lit(None).cast("int").alias(
            "week_of_year"
        ),
        F.lit(None).cast("int").alias(
            "day_of_week_number"
        ),
        F.lit("Não identificado").alias(
            "day_of_week_name"
        ),
        F.lit(None).cast("boolean").alias(
            "is_weekend"
        )
    )
)

dim_data = (
    dim_data_desconhecida
    .unionByName(
        dim_data_negocio
    )
    .withColumn(
        "_gold_run_id",
        F.lit(GOLD_RUN_ID)
    )
    .withColumn(
        "_gold_processed_at_utc",
        F.lit(GOLD_TIMESTAMP)
    )
)

salvar_tabela_gold(
    dim_data,
    "dim_data"
)

display(
    dim_data.orderBy("date_key")
)

In [0]:
# ============================================================
# VALIDAÇÃO DAS DIMENSÕES GOLD
# ============================================================

configuracao_dimensoes = [
    (
        "dim_data",
        "date_key"
    ),
    (
        "dim_clientes",
        "customer_key"
    ),
    (
        "dim_produtos",
        "product_key"
    ),
    (
        "dim_vendedores",
        "seller_key"
    ),
    (
        "dim_canais",
        "channel_key"
    ),
    (
        "dim_regioes",
        "region_key"
    )
]

resultado_validacao_dimensoes = []

for nome_tabela, chave in configuracao_dimensoes:

    dataframe = spark.table(
        f"`{CATALOG}`."
        f"`{SCHEMA_GOLD}`."
        f"`{nome_tabela}`"
    )

    quantidade = dataframe.count()

    duplicidades = (
        dataframe
        .groupBy(chave)
        .count()
        .filter(
            F.col("count") > 1
        )
        .count()
    )

    chaves_nulas = (
        dataframe
        .filter(
            F.col(chave).isNull()
        )
        .count()
    )

    membro_desconhecido = (
        dataframe
        .filter(
            F.col(chave).cast("string") == "0"
        )
        .count()
    )

    status = (
        "SUCESSO"
        if (
            duplicidades == 0
            and chaves_nulas == 0
            and membro_desconhecido == 1
        )
        else "ERRO"
    )

    resultado_validacao_dimensoes.append(
        (
            nome_tabela,
            quantidade,
            duplicidades,
            chaves_nulas,
            membro_desconhecido,
            status
        )
    )

schema_validacao_dimensoes = """
    tabela STRING,
    quantidade LONG,
    duplicidades LONG,
    chaves_nulas LONG,
    membro_nao_identificado LONG,
    status STRING
"""

df_validacao_dimensoes = (
    spark.createDataFrame(
        resultado_validacao_dimensoes,
        schema=schema_validacao_dimensoes
    )
    .orderBy("tabela")
)

display(df_validacao_dimensoes)

In [0]:
# ============================================================
# FUNÇÕES AUXILIARES — FATOS GOLD
# ============================================================

# Converte uma data para chave no formato yyyyMMdd
def gerar_chave_data(coluna):
    return (
        F.when(
            coluna.isNull(),
            F.lit(0)
        )
        .otherwise(
            F.date_format(
                coluna.cast("date"),
                "yyyyMMdd"
            ).cast("int")
        )
    )

# Valida quantidade, duplicidades e nulidade das chaves de um fato Gold
def validar_fato_gold(
    dataframe,
    nome_tabela,
    chaves,
    quantidade_esperada
):

    quantidade = dataframe.count()

    duplicidades = (
        dataframe
        .groupBy(*chaves)
        .count()
        .filter(
            F.col("count") > 1
        )
        .count()
    )

    condicao_chave_nula = reduce(
        lambda anterior, atual: anterior | atual,
        [
            F.col(chave).isNull()
            for chave in chaves
        ]
    )

    chaves_nulas = (
        dataframe
        .filter(condicao_chave_nula)
        .count()
    )

    print(f"Tabela: {nome_tabela}")
    print(f"Quantidade esperada: {quantidade_esperada}")
    print(f"Quantidade obtida: {quantidade}")
    print(f"Duplicidades: {duplicidades}")
    print(f"Chaves nulas: {chaves_nulas}")

    if quantidade != quantidade_esperada:
        raise RuntimeError(
            f"{nome_tabela}: quantidade divergente!"
        )

    if duplicidades > 0:
        raise RuntimeError(
            f"{nome_tabela}: existem chaves duplicadas!"
        )

    if chaves_nulas > 0:
        raise RuntimeError(
            f"{nome_tabela}: existem chaves nulas!"
        )

    print(f"{nome_tabela}: validação concluída com sucesso\n")


print("Funções dos fatos Gold disponíveis")

In [0]:
# ============================================================
# FATO GOLD — PEDIDOS
# ============================================================

pedidos_gold_base = (
    silver_pedidos
    .withColumn(
        "_has_order_dq",
        expressao_tem_dq(
            silver_pedidos
        )
    )
    .alias("pedido")
)

clientes_lookup = (
    dim_clientes
    .select(
        "customer_id",
        "customer_key"
    )
    .filter(
        F.col("customer_key") != "0"
    )
    .alias("cliente")
)

vendedores_lookup = (
    dim_vendedores
    .select(
        "seller_id",
        "seller_key",
        "channel_key",
        "region_key"
    )
    .filter(
        F.col("seller_key") != "0"
    )
    .alias("vendedor")
)

fct_pedidos = (
    pedidos_gold_base
    .join(
        clientes_lookup,
        F.col("pedido.customer_id")
        == F.col("cliente.customer_id"),
        "left"
    )
    .join(
        vendedores_lookup,
        F.col("pedido.seller_id")
        == F.col("vendedor.seller_id"),
        "left"
    )
    .select(
        gerar_chave_tecnica(
            F.col("pedido.order_id")
        ).alias(
            "order_key"
        ),

        F.col("pedido.order_id"),

        F.coalesce(
            F.col("cliente.customer_key"),
            F.lit("0")
        ).alias(
            "customer_key"
        ),

        F.coalesce(
            F.col("vendedor.seller_key"),
            F.lit("0")
        ).alias(
            "seller_key"
        ),

        F.coalesce(
            F.col("vendedor.channel_key"),
            F.lit("0")
        ).alias(
            "channel_key"
        ),

        F.coalesce(
            F.col("vendedor.region_key"),
            F.lit("0")
        ).alias(
            "region_key"
        ),

        gerar_chave_data(
            F.col("pedido.order_date")
        ).alias(
            "order_date_key"
        ),

        gerar_chave_data(
            F.col("pedido.promised_date")
        ).alias(
            "promised_date_key"
        ),

        F.col("pedido.order_date"),
        F.col("pedido.promised_date"),
        F.col("pedido.order_status"),

        F.col("pedido.gross_amount"),
        F.col("pedido.discount_amount"),
        F.col("pedido.net_amount"),
        F.col("pedido.financial_difference"),

        F.col("pedido.payment_priority"),
        F.col("pedido.payment_source"),

        F.lit(1).cast("long").alias(
            "order_count"
        ),

        F.when(
            F.col("pedido.order_status").isin(
                "CANCELADO",
                "CANCELLED",
                "CANCELED"
            ),
            F.lit(1)
        )
        .otherwise(
            F.lit(0)
        )
        .cast("long")
        .alias(
            "cancelled_order_count"
        ),

        F.when(
            F.col("pedido.order_status").isin(
                "ENTREGUE",
                "FATURADO"
            ),
            F.lit(1)
        )
        .otherwise(
            F.lit(0)
        )
        .cast("long")
        .alias(
            "completed_order_count"
        ),

        F.col(
            "pedido.dq_financial_mismatch"
        ),

        (
            F.coalesce(
                F.col("pedido._has_order_dq"),
                F.lit(False)
            )
            |
            F.col("cliente.customer_key").isNull()
            |
            F.col("vendedor.seller_key").isNull()
        ).alias(
            "has_data_quality_issue"
        ),

        F.lit(GOLD_RUN_ID).alias(
            "_gold_run_id"
        ),

        F.lit(GOLD_TIMESTAMP).alias(
            "_gold_processed_at_utc"
        )
    )
)

validar_fato_gold(
    dataframe=fct_pedidos,
    nome_tabela="fct_pedidos",
    chaves=["order_key"],
    quantidade_esperada=silver_pedidos.count()
)

salvar_tabela_gold(
    fct_pedidos,
    "fct_pedidos"
)

display(
    fct_pedidos
    .orderBy("order_id")
)

In [0]:
# ============================================================
# FATO GOLD — ITENS DOS PEDIDOS
# ============================================================

itens_gold_base = (
    silver_pedidos_itens
    .withColumn(
        "_has_item_dq",
        expressao_tem_dq(
            silver_pedidos_itens
        )
    )
    .alias("item")
)

produtos_lookup = (
    dim_produtos
    .select(
        "product_id",
        "product_key"
    )
    .filter(
        F.col("product_key") != "0"
    )
    .alias("produto")
)

pedidos_lookup = (
    fct_pedidos
    .select(
        "order_id",
        "order_key",
        "customer_key",
        "seller_key",
        "channel_key",
        "region_key",
        "order_date_key",
        "promised_date_key",
        "has_data_quality_issue"
    )
    .alias("pedido")
)

fct_pedidos_itens = (
    itens_gold_base
    .join(
        pedidos_lookup,
        F.col("item.order_id")
        == F.col("pedido.order_id"),
        "left"
    )
    .join(
        produtos_lookup,
        F.col("item.product_id")
        == F.col("produto.product_id"),
        "left"
    )
    .select(
        gerar_chave_tecnica(
            F.concat_ws(
                "|",
                F.col("item.order_id"),
                F.col("item.item_seq").cast("string")
            )
        ).alias(
            "order_item_key"
        ),

        F.coalesce(
            F.col("pedido.order_key"),
            F.lit("0")
        ).alias(
            "order_key"
        ),

        F.col("item.order_id"),
        F.col("item.item_seq"),

        F.coalesce(
            F.col("produto.product_key"),
            F.lit("0")
        ).alias(
            "product_key"
        ),

        F.col("item.product_id"),

        F.coalesce(
            F.col("pedido.customer_key"),
            F.lit("0")
        ).alias(
            "customer_key"
        ),

        F.coalesce(
            F.col("pedido.seller_key"),
            F.lit("0")
        ).alias(
            "seller_key"
        ),

        F.coalesce(
            F.col("pedido.channel_key"),
            F.lit("0")
        ).alias(
            "channel_key"
        ),

        F.coalesce(
            F.col("pedido.region_key"),
            F.lit("0")
        ).alias(
            "region_key"
        ),

        F.coalesce(
            F.col("pedido.order_date_key"),
            F.lit(0)
        ).alias(
            "order_date_key"
        ),

        F.col("item.quantity"),
        F.col("item.unit_price"),
        F.col("item.calculated_total_item"),
        F.col("item.total_item"),
        F.col("item.item_total_difference"),

        F.lit(1).cast("long").alias(
            "item_count"
        ),

        F.col(
            "item.dq_item_total_mismatch"
        ),

        (
            F.coalesce(
                F.col("item._has_item_dq"),
                F.lit(False)
            )
            |
            F.coalesce(
                F.col(
                    "pedido.has_data_quality_issue"
                ),
                F.lit(True)
            )
            |
            F.col("produto.product_key").isNull()
        ).alias(
            "has_data_quality_issue"
        ),

        F.lit(GOLD_RUN_ID).alias(
            "_gold_run_id"
        ),

        F.lit(GOLD_TIMESTAMP).alias(
            "_gold_processed_at_utc"
        )
    )
)

validar_fato_gold(
    dataframe=fct_pedidos_itens,
    nome_tabela="fct_pedidos_itens",
    chaves=["order_item_key"],
    quantidade_esperada=silver_pedidos_itens.count()
)

salvar_tabela_gold(
    fct_pedidos_itens,
    "fct_pedidos_itens"
)

display(
    fct_pedidos_itens
    .orderBy(
        "order_id",
        "item_seq"
    )
)

In [0]:
# ============================================================
# FATO GOLD — ENTREGAS
# ============================================================

entregas_gold_base = (
    silver_entregas
    .withColumn(
        "_has_delivery_dq",
        expressao_tem_dq(
            silver_entregas
        )
    )
    .alias("entrega")
)

contexto_pedidos_entregas = (
    fct_pedidos
    .select(
        "order_id",
        "order_key",
        "customer_key",
        "seller_key",
        "channel_key",
        "region_key",
        "order_date_key",
        "promised_date_key",
        "promised_date",
        "has_data_quality_issue"
    )
    .alias("pedido")
)

fct_entregas = (
    entregas_gold_base
    .join(
        contexto_pedidos_entregas,
        F.col("entrega.order_id")
        == F.col("pedido.order_id"),
        "left"
    )
    .select(
        gerar_chave_tecnica(
            F.col("entrega.delivery_id")
        ).alias(
            "delivery_key"
        ),

        F.col("entrega.delivery_id"),

        F.coalesce(
            F.col("pedido.order_key"),
            F.lit("0")
        ).alias(
            "order_key"
        ),

        F.col("entrega.order_id"),

        F.coalesce(
            F.col("pedido.customer_key"),
            F.lit("0")
        ).alias(
            "customer_key"
        ),

        F.coalesce(
            F.col("pedido.seller_key"),
            F.lit("0")
        ).alias(
            "seller_key"
        ),

        F.coalesce(
            F.col("pedido.channel_key"),
            F.lit("0")
        ).alias(
            "channel_key"
        ),

        F.coalesce(
            F.col("pedido.region_key"),
            F.lit("0")
        ).alias(
            "region_key"
        ),

        F.coalesce(
            F.col("pedido.order_date_key"),
            F.lit(0)
        ).alias(
            "order_date_key"
        ),

        F.coalesce(
            F.col("pedido.promised_date_key"),
            F.lit(0)
        ).alias(
            "promised_date_key"
        ),

        gerar_chave_data(
            F.to_date(
                F.col("entrega.shipped_at")
            )
        ).alias(
            "shipped_date_key"
        ),

        gerar_chave_data(
            F.to_date(
                F.col("entrega.delivered_at")
            )
        ).alias(
            "delivered_date_key"
        ),

        F.col("entrega.carrier"),
        F.col("entrega.delivery_status"),
        F.col("entrega.shipped_at"),
        F.col("entrega.delivered_at"),
        F.col("entrega.delivery_cost"),

        F.when(
            F.col("entrega.shipped_at").isNotNull()
            & F.col("entrega.delivered_at").isNotNull(),
            F.datediff(
                F.to_date(
                    F.col("entrega.delivered_at")
                ),
                F.to_date(
                    F.col("entrega.shipped_at")
                )
            )
        )
        .cast("long")
        .alias(
            "delivery_duration_days"
        ),

        F.when(
            F.col("entrega.delivered_at").isNotNull()
            & F.col("pedido.promised_date").isNotNull(),
            F.datediff(
                F.to_date(
                    F.col("entrega.delivered_at")
                ),
                F.col("pedido.promised_date")
            )
        )
        .cast("long")
        .alias(
            "delivery_delay_days"
        ),

        F.when(
            F.col("entrega.delivered_at").isNotNull()
            & F.col("pedido.promised_date").isNotNull()
            & (
                F.to_date(
                    F.col("entrega.delivered_at")
                )
                >
                F.col("pedido.promised_date")
            ),
            F.lit(1)
        )
        .otherwise(
            F.lit(0)
        )
        .cast("long")
        .alias(
            "late_delivery_count"
        ),

        F.when(
            F.col("entrega.delivery_status").isin(
                "DELIVERED",
                "ENTREGUE"
            )
            |
            F.col("entrega.delivered_at").isNotNull(),
            F.lit(1)
        )
        .otherwise(
            F.lit(0)
        )
        .cast("long")
        .alias(
            "completed_delivery_count"
        ),

        F.lit(1).cast("long").alias(
            "delivery_count"
        ),

        (
            F.coalesce(
                F.col("entrega._has_delivery_dq"),
                F.lit(False)
            )
            |
            F.coalesce(
                F.col(
                    "pedido.has_data_quality_issue"
                ),
                F.lit(True)
            )
            |
            F.col("pedido.order_key").isNull()
        ).alias(
            "has_data_quality_issue"
        ),

        F.lit(GOLD_RUN_ID).alias(
            "_gold_run_id"
        ),

        F.lit(GOLD_TIMESTAMP).alias(
            "_gold_processed_at_utc"
        )
    )
)

validar_fato_gold(
    dataframe=fct_entregas,
    nome_tabela="fct_entregas",
    chaves=["delivery_key"],
    quantidade_esperada=silver_entregas.count()
)

salvar_tabela_gold(
    fct_entregas,
    "fct_entregas"
)

display(
    fct_entregas
    .orderBy("delivery_id")
)

In [0]:
# ============================================================
# FATO GOLD — OCORRÊNCIAS
# ============================================================

ocorrencias_gold_base = (
    silver_ocorrencias
    .withColumn(
        "_has_occurrence_dq",
        expressao_tem_dq(
            silver_ocorrencias
        )
    )
    .alias("ocorrencia")
)

contexto_pedidos_ocorrencias = (
    fct_pedidos
    .select(
        "order_id",
        "order_key",
        "customer_key",
        "seller_key",
        "channel_key",
        "region_key",
        "order_date_key",
        "has_data_quality_issue"
    )
    .alias("pedido")
)

fct_ocorrencias = (
    ocorrencias_gold_base
    .join(
        contexto_pedidos_ocorrencias,
        F.col("ocorrencia.order_id")
        == F.col("pedido.order_id"),
        "left"
    )
    .select(
        gerar_chave_tecnica(
            F.col("ocorrencia.occurrence_id")
        ).alias(
            "occurrence_key"
        ),

        F.col("ocorrencia.occurrence_id"),

        F.coalesce(
            F.col("pedido.order_key"),
            F.lit("0")
        ).alias(
            "order_key"
        ),

        F.col("ocorrencia.order_id"),

        F.coalesce(
            F.col("pedido.customer_key"),
            F.lit("0")
        ).alias(
            "customer_key"
        ),

        F.coalesce(
            F.col("pedido.seller_key"),
            F.lit("0")
        ).alias(
            "seller_key"
        ),

        F.coalesce(
            F.col("pedido.channel_key"),
            F.lit("0")
        ).alias(
            "channel_key"
        ),

        F.coalesce(
            F.col("pedido.region_key"),
            F.lit("0")
        ).alias(
            "region_key"
        ),

        F.coalesce(
            F.col("pedido.order_date_key"),
            F.lit(0)
        ).alias(
            "order_date_key"
        ),

        gerar_chave_data(
            F.to_date(
                F.col("ocorrencia.opened_at")
            )
        ).alias(
            "opened_date_key"
        ),

        gerar_chave_data(
            F.to_date(
                F.col("ocorrencia.closed_at")
            )
        ).alias(
            "closed_date_key"
        ),

        F.col("ocorrencia.occurrence_status"),
        F.col("ocorrencia.category"),
        F.col("ocorrencia.subcategory"),
        F.col("ocorrencia.priority"),
        F.col("ocorrencia.service_channel"),
        F.col("ocorrencia.description"),
        F.col("ocorrencia.opened_at"),
        F.col("ocorrencia.closed_at"),
        F.col("ocorrencia.resolution_hours"),

        F.lit(1).cast("long").alias(
            "occurrence_count"
        ),

        F.when(
            F.col("ocorrencia.closed_at").isNotNull(),
            F.lit(1)
        )
        .otherwise(
            F.lit(0)
        )
        .cast("long")
        .alias(
            "closed_occurrence_count"
        ),

        (
            F.coalesce(
                F.col("ocorrencia._has_occurrence_dq"),
                F.lit(False)
            )
            |
            F.coalesce(
                F.col(
                    "pedido.has_data_quality_issue"
                ),
                F.lit(True)
            )
            |
            F.col("pedido.order_key").isNull()
        ).alias(
            "has_data_quality_issue"
        ),

        F.lit(GOLD_RUN_ID).alias(
            "_gold_run_id"
        ),

        F.lit(GOLD_TIMESTAMP).alias(
            "_gold_processed_at_utc"
        )
    )
)

validar_fato_gold(
    dataframe=fct_ocorrencias,
    nome_tabela="fct_ocorrencias",
    chaves=["occurrence_key"],
    quantidade_esperada=silver_ocorrencias.count()
)

salvar_tabela_gold(
    fct_ocorrencias,
    "fct_ocorrencias"
)

display(
    fct_ocorrencias
    .orderBy("occurrence_id")
)

## Base analítica mensal

Para não multiplicar os valores dos pedidos, entregas e ocorrências são agregadas por pedido antes do relacionamento

In [0]:
# ============================================================
# AGREGAÇÃO DAS ENTREGAS POR PEDIDO
# ============================================================

entregas_por_pedido = (
    fct_entregas
    .groupBy("order_id")
    .agg(
        F.countDistinct(
            "delivery_id"
        ).alias(
            "delivery_quantity"
        ),

        F.max(
            "completed_delivery_count"
        ).alias(
            "has_completed_delivery"
        ),

        F.max(
            "late_delivery_count"
        ).alias(
            "has_late_delivery"
        ),

        F.sum(
            "delivery_cost"
        ).alias(
            "total_delivery_cost"
        ),

        F.avg(
            "delivery_duration_days"
        ).alias(
            "average_delivery_days"
        )
    )
)

In [0]:
# ============================================================
# AGREGAÇÃO DAS OCORRÊNCIAS POR PEDIDO
# ============================================================

ocorrencias_por_pedido = (
    fct_ocorrencias
    .groupBy("order_id")
    .agg(
        F.countDistinct(
            "occurrence_id"
        ).alias(
            "occurrence_quantity"
        ),

        F.sum(
            F.when(
                F.col("resolution_hours").isNotNull(),
                F.col("resolution_hours")
            ).otherwise(
                F.lit(0.0)
            )
        ).alias(
            "resolution_hours_sum"
        ),

        F.sum(
            F.when(
                F.col("resolution_hours").isNotNull(),
                F.lit(1)
            ).otherwise(
                F.lit(0)
            )
        ).alias(
            "resolution_hours_evaluable_count"
        )
    )
)

In [0]:
# ============================================================
# BASE MENSAL — UM REGISTRO POR PEDIDO
# ============================================================

base_indicadores_mensais = (
    fct_pedidos
    .join(
        entregas_por_pedido,
        on="order_id",
        how="left"
    )
    .join(
        ocorrencias_por_pedido,
        on="order_id",
        how="left"
    )
    .withColumn(
        "month_date",
        F.trunc(
            F.col("order_date"),
            "month"
        )
    )
    .withColumn(
        "month_date_key",
        F.when(
            F.col("month_date").isNull(),
            F.lit(0)
        )
        .otherwise(
            F.date_format(
                F.col("month_date"),
                "yyyyMMdd"
            ).cast("int")
        )
    )
    .withColumn(
        "year_month",
        F.when(
            F.col("month_date").isNull(),
            F.lit("NAO_IDENTIFICADO")
        )
        .otherwise(
            F.date_format(
                F.col("month_date"),
                "yyyy-MM"
            )
        )
    )
)

In [0]:
# ============================================================
# INDICADORES MENSAIS
# ============================================================

agg_indicadores_mensais_base = (
    base_indicadores_mensais
    .groupBy(
        "month_date_key",
        "month_date",
        "year_month"
    )
    .agg(
        F.countDistinct(
            "order_id"
        ).alias(
            "order_quantity"
        ),

        F.countDistinct(
            F.when(
                F.col("customer_key") != "0",
                F.col("customer_key")
            )
        ).alias(
            "customer_quantity"
        ),

        F.sum(
            "gross_amount"
        ).alias(
            "gross_revenue"
        ),

        F.sum(
            "discount_amount"
        ).alias(
            "discount_amount"
        ),

        F.sum(
            "net_amount"
        ).alias(
            "net_revenue"
        ),

        F.sum(
            "cancelled_order_count"
        ).alias(
            "cancelled_order_quantity"
        ),

        F.sum(
            F.coalesce(
                F.col("has_completed_delivery"),
                F.lit(0)
            )
        ).alias(
            "delivered_order_quantity"
        ),

        F.sum(
            F.coalesce(
                F.col("has_late_delivery"),
                F.lit(0)
            )
        ).alias(
            "late_order_quantity"
        ),

        F.sum(
            F.coalesce(
                F.col("total_delivery_cost"),
                F.lit(0)
            )
        ).alias(
            "total_delivery_cost"
        ),

        F.avg(
            "average_delivery_days"
        ).alias(
            "average_delivery_days"
        ),

        F.sum(
            F.coalesce(
                F.col("occurrence_quantity"),
                F.lit(0)
            )
        ).alias(
            "occurrence_quantity"
        ),

        F.sum(
            F.coalesce(
                F.col("resolution_hours_sum"),
                F.lit(0.0)
            )
        ).alias(
            "_resolution_hours_sum"
        ),

        F.sum(
            F.coalesce(
                F.col(
                    "resolution_hours_evaluable_count"
                ),
                F.lit(0)
            )
        ).alias(
            "_resolution_hours_evaluable_count"
        ),

        F.sum(
            F.when(
                F.col("has_data_quality_issue"),
                F.lit(1)
            ).otherwise(
                F.lit(0)
            )
        ).alias(
            "orders_with_data_quality_issue"
        )
    )
)

In [0]:
agg_indicadores_mensais = (
    agg_indicadores_mensais_base
    .withColumn(
        "average_ticket",
        F.when(
            F.col("order_quantity") > 0,
            F.round(
                F.col("net_revenue")
                /
                F.col("order_quantity"),
                2
            )
        )
    )
    .withColumn(
        "cancellation_rate_pct",
        F.when(
            F.col("order_quantity") > 0,
            F.round(
                F.col("cancelled_order_quantity")
                /
                F.col("order_quantity")
                * F.lit(100),
                2
            )
        )
    )
    .withColumn(
        "late_delivery_rate_pct",
        F.when(
            F.col("delivered_order_quantity") > 0,
            F.round(
                F.col("late_order_quantity")
                /
                F.col("delivered_order_quantity")
                * F.lit(100),
                2
            )
        )
    )
    .withColumn(
        "average_resolution_hours",
        F.when(
            F.col(
                "_resolution_hours_evaluable_count"
            ) > 0,
            F.round(
                F.col("_resolution_hours_sum")
                /
                F.col(
                    "_resolution_hours_evaluable_count"
                ),
                2
            )
        )
    )
    .withColumn(
        "data_quality_rate_pct",
        F.when(
            F.col("order_quantity") > 0,
            F.round(
                F.col(
                    "orders_with_data_quality_issue"
                )
                /
                F.col("order_quantity")
                * F.lit(100),
                2
            )
        )
    )
    .withColumn(
        "_gold_run_id",
        F.lit(GOLD_RUN_ID)
    )
    .withColumn(
        "_gold_processed_at_utc",
        F.lit(GOLD_TIMESTAMP)
    )
    .drop(
        "_resolution_hours_sum",
        "_resolution_hours_evaluable_count"
    )
    .orderBy("month_date_key")
)

salvar_tabela_gold(
    agg_indicadores_mensais,
    "agg_indicadores_mensais"
)

display(
    agg_indicadores_mensais
)

In [0]:
# ============================================================
# AUDITORIA ESTRUTURAL DA GOLD
# ============================================================

configuracao_auditoria_gold = [
    (
        "dim_data",
        ["date_key"],
        "uma linha por data"
    ),
    (
        "dim_clientes",
        ["customer_key"],
        "uma linha por cliente"
    ),
    (
        "dim_produtos",
        ["product_key"],
        "uma linha por produto"
    ),
    (
        "dim_vendedores",
        ["seller_key"],
        "uma linha por vendedor"
    ),
    (
        "dim_canais",
        ["channel_key"],
        "uma linha por canal"
    ),
    (
        "dim_regioes",
        ["region_key"],
        "uma linha por região"
    ),
    (
        "fct_pedidos",
        ["order_key"],
        "uma linha por pedido"
    ),
    (
        "fct_pedidos_itens",
        ["order_item_key"],
        "uma linha por item do pedido"
    ),
    (
        "fct_entregas",
        ["delivery_key"],
        "uma linha por entrega"
    ),
    (
        "fct_ocorrencias",
        ["occurrence_key"],
        "uma linha por ocorrência"
    ),
    (
        "agg_indicadores_mensais",
        ["month_date_key"],
        "uma linha por mês"
    )
]

resultado_auditoria_gold = []

for (
    nome_tabela,
    chaves,
    granularidade
) in configuracao_auditoria_gold:

    dataframe = spark.table(
        f"`{CATALOG}`."
        f"`{SCHEMA_GOLD}`."
        f"`{nome_tabela}`"
    )

    quantidade = dataframe.count()

    duplicidades = (
        dataframe
        .groupBy(*chaves)
        .count()
        .filter(
            F.col("count") > 1
        )
        .count()
    )

    condicao_nula = reduce(
        lambda anterior, atual: anterior | atual,
        [
            F.col(chave).isNull()
            for chave in chaves
        ]
    )

    chaves_nulas = (
        dataframe
        .filter(condicao_nula)
        .count()
    )

    status = (
        "SUCESSO"
        if (
            quantidade > 0
            and duplicidades == 0
            and chaves_nulas == 0
        )
        else "ERRO"
    )

    resultado_auditoria_gold.append(
        (
            GOLD_RUN_ID,
            nome_tabela,
            granularidade,
            quantidade,
            duplicidades,
            chaves_nulas,
            status,
            GOLD_TIMESTAMP
        )
    )


schema_auditoria_gold = """
    gold_run_id STRING,
    tabela STRING,
    granularidade STRING,
    quantidade LONG,
    duplicidades LONG,
    chaves_nulas LONG,
    status STRING,
    processed_at_utc TIMESTAMP
"""

df_auditoria_gold = (
    spark.createDataFrame(
        resultado_auditoria_gold,
        schema=schema_auditoria_gold
    )
    .orderBy("tabela")
)

display(df_auditoria_gold)

In [0]:
# ============================================================
#  AUDITORIA GOLD
# ============================================================

TABELA_AUDITORIA_GOLD = (
    f"`{CATALOG}`."
    f"`{SCHEMA_GOLD}`."
    f"`gold_model_audit`"
)

(
    df_auditoria_gold.write
    .format("delta")
    .mode("append")
    .option(
        "mergeSchema",
        "true"
    )
    .saveAsTable(
        TABELA_AUDITORIA_GOLD
    )
)

erros_gold = (
    df_auditoria_gold
    .filter(
        F.col("status") != "SUCESSO"
    )
    .count()
)

if erros_gold > 0:
    raise RuntimeError(
        "A auditoria identificou erros estruturais na Gold"
    )

print(
    "Modelo Gold construído e auditado com sucesso!!!"
)

# Conclusão do modelo Gold

O modelo analítico final foi construído com granularidades independentes para evitar duplicação de métricas:

- `fct_pedidos`: um registro por pedido;
- `fct_pedidos_itens`: um registro por item do pedido;
- `fct_entregas`: um registro por entrega;
- `fct_ocorrencias`: um registro por ocorrência.

Foram criadas dimensões conformadas de clientes, produtos, vendedores, canais, regiões e calendário.

As dimensões possuem chaves técnicas determinísticas e um membro não identificado, utilizado para preservar fatos com relacionamentos ausentes ou inválidos.

Também foi criada a tabela `agg_indicadores_mensais`, contendo:

- quantidade de pedidos;
- clientes distintos;
- receita bruta;
- descontos;
- receita líquida;
- ticket médio;
- pedidos cancelados;
- taxa de cancelamento;
- pedidos entregues;
- pedidos entregues com atraso;
- taxa de atraso;
- custo logístico;
- tempo médio de entrega;
- ocorrências de atendimento;
- tempo médio de resolução;
- indicadores de qualidade dos dados.

Todas as tabelas Gold foram persistidas em Delta e validadas quanto à granularidade, duplicidades e chaves nulas.